In [0]:
from pyspark.sql.functions import col, sum, when, coalesce, lit

SILVER_SALES_TABLE = "workspace.retail.silver_sales"
SILVER_AUX_TABLE = "workspace.retail.silver_auxiliary"
GOLD_AGG_EXECUTIVE = "workspace.retail.gold_agg_daily_executive"

def build_daily_executive(df_sales, df_aux):
    df_sales_agg = df_sales.groupBy("invoice_date_only").agg(
        sum(when(col("total_amount") > 0, col("total_amount")).otherwise(0)).alias("gross_revenue"),
        sum(when(col("total_amount") < 0, abs(col("total_amount"))).otherwise(0)).alias("total_returns")
    )
    
    df_aux_agg = df_aux.groupBy("invoice_date_only").agg(
        sum(when(col("row_category") == "OPS_FEE", abs(col("total_amount"))).otherwise(0)).alias("ops_shipping_costs"),
        sum(when(col("row_category") == "INVENTORY_ADJUSTMENT", abs(col("total_amount"))).otherwise(0)).alias("inventory_loss_value"),
        sum(when(col("row_category") == "CUSTOMER_PROMOTION", 0).otherwise(0)).alias("marketing_promo_cost") 
    )
    
    df_final = df_sales_agg.join(
        df_aux_agg, 
        df_sales_agg.invoice_date_only == df_aux_agg.invoice_date_only, 
        "full_outer"
    ).select(
        coalesce(df_sales_agg.invoice_date_only, df_aux_agg.invoice_date_only).alias("report_date"),
        "gross_revenue",
        "total_returns",
        "ops_shipping_costs",
        "inventory_loss_value",
        "marketing_promo_cost",
    ).fillna(0) 
    
    df_final = df_final.withColumn("net_revenue", col("gross_revenue") - col("total_returns"))
    
    return df_final.orderBy("report_date")

def main():
    df_sales = spark.table(SILVER_SALES_TABLE)
    df_aux = spark.table(SILVER_AUX_TABLE)
    
    df = build_daily_executive(df_sales, df_aux)
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_AGG_EXECUTIVE)
        
    print(f"Executive table ready with {df.count()} days of data.")

if __name__ == "__main__":
    main()